# FedAudit — E4 brain MRI with a pretrained ResNet-18

Repeats the leakage-inflation experiment on the brain MRI federation with an ImageNet-pretrained
ResNet-18 (10 training methods x 3 seeds = 30 runs). The data splits and the images flagged by the
audit are embedded in this notebook and are identical to those of the main experiments.

**Inputs to attach (+ Add Input):** the four brain MRI datasets (figshare/Cheng, SARTAJ, Br35H, Nickparvar).
**Settings:** GPU T4 x2 and Internet on (the pretrained ResNet-18 weights are downloaded once).

**Run 1:** keep `QUICK_TEST = True` and Run all (about 15 min).
**Run 2:** set `QUICK_TEST = False`, Save Version → Save & Run All (Commit). To continue a stopped run,
attach this notebook's previous output as an input and commit again; finished runs are skipped.
Download `fedaudit_e4_resnet18.zip` at the end.

In [ ]:
# ============================== SETTINGS ==============================
QUICK_TEST = True          # True: short pipeline check. False: the 30 runs.
TIME_BUDGET_HOURS = 11.0   # stop starting new runs after this (Kaggle stops sessions at 12 h)
ROUNDS = 30
SEEDS = [0, 1, 2]
MODEL = "resnet18"
SOURCE_OVERRIDES = {}      # e.g. {"figshare": "/kaggle/input/my-figshare-copy"} if auto-detection misses one
# ======================================================================

In [ ]:
import base64, gzip, io, os, sys, tarfile, subprocess, time
START = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h5py"], check=False)
CODE_DIR = "/kaggle/working/fedaudit_code"
os.makedirs(CODE_DIR, exist_ok=True)
PKG = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(PKG)), mode="r:gz").extractall(CODE_DIR, filter="data")
sys.path.insert(0, CODE_DIR)
for m in [m for m in sys.modules if m == "fedaudit" or m.startswith("fedaudit.")]:
    del sys.modules[m]
import fedaudit, torch
print("fedaudit code ready |", "CUDA GPUs:", torch.cuda.device_count())

In [ ]:
# Pretrained ResNet-18 weights: downloaded once here, or copied from an attached input (resnet18*.pth)
import glob, shutil
from torchvision.models import ResNet18_Weights
ckpt_dir = os.path.join(torch.hub.get_dir(), "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
target = os.path.join(ckpt_dir, os.path.basename(ResNet18_Weights.IMAGENET1K_V1.url))
if not os.path.exists(target):
    local = sorted(glob.glob("/kaggle/input/**/resnet18*.pth", recursive=True))
    if local:
        shutil.copy(local[0], target)
    else:
        try:
            ResNet18_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
        except Exception as e:
            raise RuntimeError("Could not download the ResNet-18 weights. Turn on Internet in the notebook settings "
                               "(Settings > Internet; it needs a phone-verified Kaggle account), or attach a dataset "
                               "that contains a resnet18*.pth file.") from e
print("ResNet-18 weights ready:", target, f"{os.path.getsize(target) / 1e6:.0f} MB")

In [ ]:
# Step 1: restore the brain MRI manifest and audit pairs of the main experiments, rebuild the image cache
import pandas as pd
from fedaudit.experiments import sources
from fedaudit.experiments.prepare import restore_stores
from fedaudit.experiments.runner import merge_previous

BRAIN_CSV = "..."
PAIRS_CSV = "..."
OLD_ROOTS = {"br35h": "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection", "figshare": "/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset", "nickparvar": "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset", "sartaj": "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri"}

OUT = "/kaggle/working/fedaudit_e4_resnet18_quick" if QUICK_TEST else "/kaggle/working/fedaudit_e4_resnet18"
for d in ("data", "results", "runs"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
S = dict(out=OUT, input_root="/kaggle/input", code_dir=CODE_DIR, source_overrides=SOURCE_OVERRIDES,
         quick=1 if QUICK_TEST else 0, size=128, rounds=2 if QUICK_TEST else ROUNDS,
         seeds=SEEDS[:1] if QUICK_TEST else SEEDS, model=MODEL, experiments=["E4brain"],
         deadline=START + TIME_BUDGET_HOURS * 3600)

src = sources.detect_sources(S["input_root"], SOURCE_OVERRIDES)
print("sources found:", src)
df = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(BRAIN_CSV))))
for name, old in OLD_ROOTS.items():  # datasets may be mounted under a different folder in this session
    if name in src and old != src[name]:
        m = df["path"].str.startswith(old)
        df.loc[m, "path"] = src[name] + df.loc[m, "path"].str[len(old):]
df.to_csv(f"{OUT}/data/brain.csv", index=False)
open(f"{OUT}/results/audit_pairs_brain.csv", "wb").write(gzip.decompress(base64.b64decode(PAIRS_CSV)))
print(f"Merged {merge_previous(S)} finished runs from previous versions")
restore_stores(S, src)

In [ ]:
# Step 2: 30 training runs on both GPUs
from fedaudit.experiments.runner import run_all
run_all(S, n_gpus=max(1, torch.cuda.device_count()))

In [ ]:
# Step 3: zip the runs
import glob, zipfile, shutil
name = "fedaudit_e4_resnet18_quick" if QUICK_TEST else "fedaudit_e4_resnet18"
with zipfile.ZipFile(f"/kaggle/working/{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{OUT}/**/*", recursive=True):
        if os.path.isfile(p) and "/store/" not in p and "/claims/" not in p:
            z.write(p, os.path.join("fedaudit_e4_resnet18", os.path.relpath(p, OUT)))
shutil.rmtree(f"{OUT}/store", ignore_errors=True)
print(len(glob.glob(f"{OUT}/runs/*.json")), "runs finished |", "errors:", len(glob.glob(f"{OUT}/errors/*")))
print("Wrote", f"/kaggle/working/{name}.zip")